In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import math
import time
import shapely
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree
from pyproj import CRS, Transformer

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "processed" / "student_test").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find PathSnap project root. Open this notebook inside the PathSnap project.")

DATA_DIR = PROJECT_ROOT / "data" / "processed"
TEST_DIR = DATA_DIR / "student_test"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "candidates"

assert TEST_DIR.exists(), f"Missing {TEST_DIR}"
assert int(shapely.__version__.split(".")[0]) >= 2, "Install Shapely 2.x with: %pip install --upgrade shapely"

print("Project:", PROJECT_ROOT)
print("Student test:", TEST_DIR)
print("Output:", OUTPUT_DIR)
print("Shapely:", shapely.__version__)

Project: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap
Student test: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\student_test
Output: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\interim\candidates
Shapely: 2.2.0


In [2]:
TRACK_LON, TRACK_LAT, TRACK_TIME = "longitude", "latitude", "timestamp_s"
NODE_LON, NODE_LAT = "longitude", "latitude"
ARC_FROM, ARC_TO = "from_node_id", "to_node_id"
RADII_M = [30.0, 60.0, 100.0]
MAX_CANDIDATES = 20
OVERWRITE = False

def find_trip_dirs(partition_dir):
    required = {"track.csv", "nodes.csv", "arcs.csv", "route.csv"}
    return sorted(
        p for p in partition_dir.iterdir()
        if p.is_dir() and required.issubset({f.name for f in p.iterdir() if f.is_file()})
    )

def load_trip(folder):
    return {
        "track": pd.read_csv(folder / "track.csv"),
        "nodes": pd.read_csv(folder / "nodes.csv"),
        "arcs": pd.read_csv(folder / "arcs.csv"),
        "route": pd.read_csv(folder / "route.csv"),
    }

test_dirs = find_trip_dirs(TEST_DIR)
print("Student-test trips found:", len(test_dirs))
if len(test_dirs) == 0:
    raise ValueError(f"No trip folders containing track.csv, nodes.csv, arcs.csv and route.csv were found in {TEST_DIR}")

for folder in test_dirs[:3]:
    print(folder.name)
if len(test_dirs) != 10:
    print("WARNING: Expected 10 student-test trips; found", len(test_dirs))

Student-test trips found: 10
00000000
00000007
00000009


In [3]:
def make_local_transformer(track):
    lon0 = float(track[TRACK_LON].median())
    lat0 = float(track[TRACK_LAT].median())
    local_crs = CRS.from_proj4(
        f"+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m +no_defs"
    )
    transformer = Transformer.from_crs("EPSG:4326", local_crs, always_xy=True)
    return transformer

def project_xy(frame, lon_col, lat_col, transformer):
    x, y = transformer.transform(
        frame[lon_col].to_numpy(dtype=float),
        frame[lat_col].to_numpy(dtype=float)
    )
    return np.asarray(x), np.asarray(y)

def build_arc_geometries(nodes, arcs, node_x, node_y):
    src = pd.to_numeric(arcs[ARC_FROM], errors="coerce").to_numpy()
    dst = pd.to_numeric(arcs[ARC_TO], errors="coerce").to_numpy()

    if not np.isfinite(src).all() or not np.isfinite(dst).all():
        raise ValueError("Arc endpoint IDs contain non-numeric or non-finite values.")
    src = src.astype(np.int64)
    dst = dst.astype(np.int64)

    if (src < 0).any() or (dst < 0).any() or (src >= len(nodes)).any() or (dst >= len(nodes)).any():
        raise ValueError("Arc endpoint ID outside node row-position range.")

    geometries = []
    meta = []
    for arc_id, (u, v) in enumerate(zip(src, dst)):
        x1, y1 = node_x[u], node_y[u]
        x2, y2 = node_x[v], node_y[v]
        if not np.isfinite([x1, y1, x2, y2]).all():
            continue
        line = LineString([(x1, y1), (x2, y2)])
        if line.length <= 0:
            continue
        dx, dy = x2 - x1, y2 - y1
        bearing = (math.degrees(math.atan2(dx, dy)) + 360.0) % 360.0
        geometries.append(line)
        meta.append((arc_id, float(line.length), float(bearing), int(u), int(v)))

    meta_df = pd.DataFrame(meta, columns=["arc_id", "segment_length_m", "road_heading_deg", "from_node_id", "to_node_id"])
    return geometries, meta_df

def circular_difference_deg(a, b):
    return np.abs((a - b + 180.0) % 360.0 - 180.0)

def gps_headings(x, y):
    n = len(x)
    headings = np.full(n, np.nan, dtype=float)
    for i in range(n):
        if i < n - 1:
            dx, dy = x[i + 1] - x[i], y[i + 1] - y[i]
        elif i > 0:
            dx, dy = x[i] - x[i - 1], y[i] - y[i - 1]
        else:
            continue
        if math.hypot(dx, dy) > 0.5:
            headings[i] = (math.degrees(math.atan2(dx, dy)) + 360.0) % 360.0
    return headings

In [4]:
def generate_candidates(track, geometries, arc_meta, transformer,
                        radii_m=RADII_M, max_candidates=MAX_CANDIDATES):
    if len(geometries) == 0:
        raise ValueError("This trip has no usable road arc geometries.")

    tree = STRtree(geometries)
    tx, ty = project_xy(track, TRACK_LON, TRACK_LAT, transformer)
    headings = gps_headings(tx, ty)
    meta = arc_meta.set_index("arc_id")
    rows = []
    radius = max(radii_m)

    for obs_pos, (x, y) in enumerate(zip(tx, ty)):
        point = Point(float(x), float(y))
        idx = np.asarray(tree.query(point, predicate="dwithin", distance=radius), dtype=int)
        used_fallback = len(idx) == 0
        if used_fallback:
            idx = np.array([int(tree.nearest(point))])

        scored = []
        for geom_idx in idx:
            line = geometries[int(geom_idx)]
            distance = float(point.distance(line))
            arc_id = int(arc_meta.iloc[int(geom_idx)]["arc_id"])
            scored.append((distance, arc_id, line, int(geom_idx)))

        scored.sort(key=lambda item: item[0])
        scored = scored[:max_candidates]

        for rank, (distance, arc_id, line, geom_idx) in enumerate(scored, start=1):
            m = meta.loc[arc_id]
            projection = line.project(point)
            fraction = projection / line.length if line.length else 0.0
            projected_point = line.interpolate(projection)
            hdiff = (
                float(circular_difference_deg(headings[obs_pos], m["road_heading_deg"]))
                if np.isfinite(headings[obs_pos]) else np.nan
            )
            rows.append({
                "observation_pos": obs_pos,
                "observation_id": track.iloc[obs_pos].get("observation_id", obs_pos),
                "arc_id": arc_id,
                "candidate_rank": rank,
                "distance_m": distance,
                "distance_to_projected_point_m": float(point.distance(projected_point)),
                "segment_length_m": float(m["segment_length_m"]),
                "gps_heading_deg": headings[obs_pos],
                "road_heading_deg": float(m["road_heading_deg"]),
                "heading_difference_deg": hdiff,
                "projection_fraction": float(fraction),
                "used_fallback": bool(used_fallback),
                "search_radius_m": radius,
            })

    return pd.DataFrame(rows)

def process_trip(folder):
    data = load_trip(folder)
    transformer = make_local_transformer(data["track"])
    node_x, node_y = project_xy(data["nodes"], NODE_LON, NODE_LAT, transformer)
    geometries, arc_meta = build_arc_geometries(data["nodes"], data["arcs"], node_x, node_y)
    candidates = generate_candidates(data["track"], geometries, arc_meta, transformer)
    candidates.insert(0, "trip_id", folder.name)
    return candidates

In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_dir = OUTPUT_DIR / "student_test"
out_dir.mkdir(parents=True, exist_ok=True)
summaries = []

for i, folder in enumerate(test_dirs, start=1):
    t0 = time.time()
    out_path = out_dir / f"trip_{folder.name}_candidates.csv"

    if out_path.exists() and not OVERWRITE:
        candidates = pd.read_csv(out_path)
        print(f"[{i}/{len(test_dirs)}] {folder.name}: already exists, skipped")
    else:
        candidates = process_trip(folder)
        candidates.to_csv(out_path, index=False)
        print(f"[{i}/{len(test_dirs)}] {folder.name}: {len(candidates):,} candidate rows saved")

    obs_count = candidates["observation_pos"].nunique()
    fallback_count = int(candidates.groupby("observation_pos")["used_fallback"].first().sum())
    summaries.append({
        "trip_id": folder.name,
        "candidate_rows": len(candidates),
        "observations": obs_count,
        "fallback_observations": fallback_count,
        "runtime_seconds": time.time() - t0,
        "output_file": str(out_path),
    })

student_test_summary = pd.DataFrame(summaries)
student_test_summary.to_csv(OUTPUT_DIR / "student_test_candidate_summary.csv", index=False)
display(student_test_summary)

print("Total trips:", len(student_test_summary))
print("Total observations:", student_test_summary["observations"].sum())
print("Total candidate rows:", student_test_summary["candidate_rows"].sum())
print("Total fallback observations:", student_test_summary["fallback_observations"].sum())
print("Total runtime:", round(student_test_summary["runtime_seconds"].sum(), 2), "seconds")
print("Saved to:", out_dir)

[1/10] 00000000: 19,892 candidate rows saved
[2/10] 00000007: 9,305 candidate rows saved
[3/10] 00000009: 19,985 candidate rows saved
[4/10] 00000040: 25,458 candidate rows saved
[5/10] 00000052: 4,299 candidate rows saved
[6/10] 00000056: 67,054 candidate rows saved
[7/10] 00000065: 14,852 candidate rows saved
[8/10] 00000076: 29,104 candidate rows saved
[9/10] 00000077: 69,967 candidate rows saved
[10/10] 00000091: 15,509 candidate rows saved


,trip_id,candidate_rows,observations,fallback_observations,runtime_seconds,output_file
0,00000000,19892,2503,0,1.757233,c:\Users\HP\.vscode\python\ml\mlcodes\ml_proje...
1,00000007,9305,1354,0,0.689450,c:\Users\HP\.vscode\python\ml\mlcodes\ml_proje...
2,00000009,19985,1049,0,1.923251,c:\Users\HP\.vscode\python\ml\mlcodes\ml_proje...
3,00000040,25458,2995,0,2.097436,c:\Users\HP\.vscode\python\ml\mlcodes\ml_proje...
4,00000052,4299,262,0,0.347848,c:\Users\HP\.vscode\python\ml\mlcodes\ml_proje...
5,00000056,67054,3958,0,5.838842,c:\Users\HP\.vscode\python\ml\mlcodes\ml_proje...
6,00000065,14852,768,0,1.426966,c:\Users\HP\.vscode\python\ml\mlcodes\ml_proje...
7,00000076,29104,1486,0,4.126667,c:\Users\HP\.vscode\python\ml\mlcodes\ml_proje...
8,00000077,69967,3911,0,8.089519,c:\Users\HP\.vscode\python\ml\mlcodes\ml_proje...
9,00000091,15509,874,0,2.043885,c:\Users\HP\.vscode\python\ml\mlcodes\ml_proje...


Total trips: 10
Total observations: 19160
Total candidate rows: 275425
Total fallback observations: 0
Total runtime: 28.34 seconds
Saved to: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\interim\candidates\student_test


## Output

The notebook writes one candidate CSV per student-test trip to `data/interim/candidates/student_test/`, plus `student_test_candidate_summary.csv`. Existing files are skipped by default. Set `OVERWRITE = True` only if you want to regenerate them.